# 02 — VOC to YOLO WBC preprocessing

**Purpose:** Create an idempotent, single-class WBC training dataset with frozen, leakage-free splits.

**Required inputs:** Successful notebook 01; configured BCCD source.

**Saved outputs:** `data/processed/BCCD_YOLO/`, `outputs/preprocessing_report.json`, split manifests and before/after figure.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

## 2. Convert and preserve splits

Only WBC objects become class 0. Coordinates are clipped to actual image bounds and transformed with continuous-edge VOC formulas. Source files are never changed.

In [ ]:
from src.preprocessing import preprocess, conversion_table
report = preprocess(cfg)
display(report)
print("Split policy:", report["split_reason"])
print("Clipped boxes:", len(report["clipped_boxes"]))

## 3. Inspect round-trip and before/after

The table shows source VOC coordinates, normalized YOLO values and reversible pixel error.

In [ ]:
from src.bccd import inspect_dataset
from src.visualization import before_after
records = inspect_dataset(cfg)["records"]
selected = next((r for r in records if any(o["class"] == "WBC" for o in r["objects"])), None)
assert selected is not None, "No WBC examples in source"
table = conversion_table(selected)
display(table)
assert (table["roundtrip_max_error_px"] < 1e-4).all()
display(before_after(selected, cfg["processed_data_path"], out / "figures" / "preprocessing_examples.png"))

## 4. Verify saved dataset

This checks labels, source provenance, disjoint IDs and cross-split image hashes.

In [ ]:
from src.validation import validate_processed
validation = validate_processed(cfg["processed_data_path"], out / "validation_report.json")
display(pd.DataFrame(validation["splits"]).T if "pd" in globals() else validation["splits"])
assert validation["ready"] and (out / "splits" / "manifest.json").is_file()
print("PASS — processed WBC dataset ready. Next: notebook 03.")

## Next notebook

Open **03_Dataset_Visualization.ipynb**.